# WorkWise PH — Exploratory Data Analysis

Runs against `clean.fact_long` using the project engine (reads `DATABASE_URL` from `backend/.env`).
Run from the repo root: `jupyter nbconvert --to notebook --execute data_pipeline/notebooks/EDA_REPORT.ipynb`.
Narrative answers live in `docs/EDA_FINDINGS.md`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1] if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()))
import pandas as pd
from backend.app.db.session import engine

def q(sql):
    return pd.read_sql(sql, engine)

## Highest unemployment & underemployment years (annual avg, Both Sexes)

In [ ]:
q("""SELECT indicator_name, year, round(avg(value)::numeric,2) AS avg_rate
      FROM clean.fact_long
      WHERE source_table='raw.lfs_rates' AND sex='Both Sexes' AND period_type='monthly'
        AND indicator_name IN ('Unemployment Rate','Underemployment Rate') AND value IS NOT NULL
      GROUP BY indicator_name, year ORDER BY indicator_name, avg_rate DESC""")

## Labor-force participation gap by sex

In [ ]:
q("""SELECT sex, round(avg(value)::numeric,1) AS avg_lfpr
      FROM clean.fact_long
      WHERE source_table='raw.lfs_rates' AND indicator_name='Labor Force Participation Rate'
        AND sex IN ('Male','Female') AND period_type='monthly' AND value IS NOT NULL
      GROUP BY sex""")

## Largest industries (latest period)

In [ ]:
q("""SELECT category, round(value::numeric,0) AS thousands
      FROM clean.fact_long
      WHERE source_table='raw.employed_industry_2009' AND category <> 'TOTAL' AND value IS NOT NULL
        AND reference_date=(SELECT max(reference_date) FROM clean.fact_long
                            WHERE source_table='raw.employed_industry_2009' AND value IS NOT NULL)
      ORDER BY thousands DESC LIMIT 8""")

## Underemployment share by education (latest)

In [ ]:
q("""WITH e AS (SELECT category, value FROM clean.fact_long
             WHERE source_table='raw.education_employed' AND value IS NOT NULL
               AND reference_date=(SELECT max(reference_date) FROM clean.fact_long WHERE source_table='raw.education_employed' AND value IS NOT NULL)),
          u AS (SELECT category, value FROM clean.fact_long
             WHERE source_table='raw.education_underemployed' AND value IS NOT NULL
               AND reference_date=(SELECT max(reference_date) FROM clean.fact_long WHERE source_table='raw.education_underemployed' AND value IS NOT NULL))
     SELECT e.category, round((u.value/e.value*100)::numeric,1) AS underemp_share_pct
     FROM e JOIN u USING(category) WHERE e.category<>'Total' AND e.value>0
     ORDER BY underemp_share_pct DESC""")